In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from collections import Counter
import numpy as np
from torchvision import models
import sys
import os
sys.path.append(os.path.abspath(".."))
from augmenter.custom_augment import CustomAugmenter

In [4]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda:1


In [5]:
transform = transforms.Compose([
    CustomAugmenter(),                   
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])  
])

dataset_path = "../dataset"
full_dataset = datasets.ImageFolder(root=dataset_path, transform=transform)
class_names = full_dataset.classes
num_classes = len(class_names)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

def count_classes(dataset, full_dataset):
    indices = dataset.indices if isinstance(dataset, torch.utils.data.Subset) else range(len(dataset))
    labels = [full_dataset[i][1] for i in indices]
    return Counter(labels)

train_class_counts = count_classes(train_dataset, full_dataset)
val_class_counts = count_classes(val_dataset, full_dataset)

print("\nClass distribution in training set:")
for cls_idx, count in train_class_counts.items():
    print(f"  {class_names[cls_idx]}: {count} samples")

print("\nClass distribution in validation set:")
for cls_idx, count in val_class_counts.items():
    print(f"  {class_names[cls_idx]}: {count} samples")


train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)



Class distribution in training set:
  pulse: 7947 samples
  single_tone: 7967 samples
  benign: 8205 samples
  wideband: 8053 samples

Class distribution in validation set:
  pulse: 2060 samples
  benign: 1995 samples
  wideband: 1953 samples
  single_tone: 2036 samples


In [ ]:
# Load pretrained ResNet18 and replace the final layer
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# 7. Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 8. Training loop with validation
num_epochs = 10
train_losses = []
val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    # Validation loop
    model.eval()
    total_val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_val_loss += loss.item()

            # Accuracy tracking
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_train = total_train_loss / len(train_loader)
    avg_val = total_val_loss / len(val_loader)
    val_acc = correct / total

    train_losses.append(avg_train)
    val_losses.append(avg_val)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} | Val Accuracy: {val_acc:.4f}")

# 9. Plot training and validation loss
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

# 10. Plot validation accuracy
plt.subplot(1, 2, 2)
plt.plot(val_accuracies, label='Val Accuracy', color='green')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 11. Confusion Matrix on Validation Set
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# 12. Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title("Confusion Matrix on Validation Set")
plt.tight_layout()
plt.show()

# 13. Classification Report
print("\n Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))